# 第11周 · 周二：数据清洗与EDA——先看透数据再动手

> 难度：⭐⭐⭐ | 定位：掌握数据清洗的系统方法论与探索性数据分析的完整流程，建立"先看数据再分析"的纪律
> 前置依赖：W11D1选题与数据获取 | 后续用途：W11D3特征工程、W11D4-5深度分析
> 面试地位：数据清洗和EDA是数据分析师日常工作中占比最高（约60%）的环节，面试中常被问到"拿到数据后你会先做什么"

---

## 区块1：核心概念与第一性原理

### 1.1 EDA的第一性原理

EDA（Exploratory Data Analysis，探索性数据分析）的本质是：**在建立任何模型或得出任何结论之前，先用统计方法和可视化手段理解数据的基本特征**。

从第一性原理推导EDA的必要性：

**原命题**：数据分析的目标是从数据中提取有价值的洞察。
**推论1**：如果数据本身有问题（缺失、异常、错误），提取的洞察必然不可靠。
**推论2**：如果不理解数据的分布和关系，就无法选择正确的分析方法。
**结论**：EDA是不可跳过的步骤——它不是"做了更好"，而是"不做就无法保证后续分析的正确性"。

### 1.2 数据质量的六个维度

数据质量不是"有没有缺失值"这么简单。业界通用的数据质量框架包含六个维度：

| 维度 | 定义 | 检查方法 | 典型问题 |
|------|------|---------|----------|
| **完整性** | 数据是否缺失 | df.isnull().sum() / len(df) | 某个字段30%为空 |
| **唯一性** | 是否有重复记录 | df.duplicated().sum() | 同一订单出现两次 |
| **一致性** | 同一含义的数据是否一致 | 跨表验证、业务规则检查 | 同一用户的注册日期在两个表里不同 |
| **准确性** | 数据是否符合真实情况 | 极值检查、范围检查 | 年龄=200岁、金额=负数 |
| **有效性** | 数据是否符合格式规范 | 正则匹配、类型检查 | 手机号格式不对、日期格式混乱 |
| **时效性** | 数据是否足够"新鲜" | 时间范围检查 | 数据只到3个月前，没有近期数据 |

### 1.3 EDA的三个层次

EDA不是"随便画几张图看看"，它有三个递进的层次：

**层次1：单变量分析**。每次只看一个变量——它的分布是什么样的？有异常值吗？数据跨度合理吗？
- 数值型变量：直方图（分布形状）、箱线图（异常值）、describe()（基本统计量）
- 分类型变量：柱状图（频数分布）、value_counts()（各类别占比）

**层次2：双变量分析**。看两个变量之间的关系——它们相关吗？是什么模式？
- 数值 vs 数值：散点图、相关系数矩阵
- 数值 vs 分类：分组箱线图、分组均值对比
- 分类 vs 分类：交叉表(crosstab)、堆叠柱状图

**层次3：多变量分析**。同时看多个变量的交互——有没有组合效应？条件分布是怎样的？
- 按多个维度分组聚合
- 热力图（按两个分类维度聚合一个数值指标）
- Pairplot（多变量两两散点图矩阵）

### 1.4 数据清洗的"20/80法则"

数据清洗中，20%的清洗动作解决了80%的数据质量问题。关键是找到这20%：

- **首先处理**：缺失率>10%的字段、明显的异常值（如负数金额）、完全重复的行
- **其次处理**：格式不一致（如日期格式混用）、命名不规范（如"北京"和"北京市"）
- **低优先级**：缺失率<1%且不影响核心分析的字段、对结论无影响的极少数异常值

### 1.5 面试高频考点

**面试官常问**：
- "拿到一份新数据后，你的第一步是什么？"
- "如何处理缺失值？有哪些策略？"
- "如何识别和处理异常值？"
- "你如何判断数据'干净'了可以开始分析？"

**回答要点**：
- 第一步是数据嗅探：行数、列数、类型、缺失值、描述统计、时间范围
- 缺失值处理策略：删除（缺失率高）、填充（均值/中位数/众数/前向填充/模型预测）、保留（缺失本身可能是一个信号）
- 异常值识别：3-sigma法则、IQR(箱线图)、业务规则；处理：截断、转换、分段处理
- "干净"的标准：缺失值已处理、异常值已标记或处理、数据类型正确、定义了明确的过滤条件

## 区块2：知识点详细拆解

### 2.1 缺失值处理策略详解

**为什么重要**：缺失值处理直接影响分析结果的准确性。错误的填充策略可能引入系统性偏差。

**策略1：删除**
```python
# 删除含有缺失值的行
df.dropna(subset=['amount'])
# 删除缺失率>50%的列
df.dropna(axis=1, thresh=len(df)*0.5)
```
- 适用场景：缺失率很低（<1%）或很高（>50%）
- 风险：删除可能引入偏差（如果缺失不是随机的）

**策略2：填充——统计量**
```python
# 均值/中位数/众数填充
df['amount'].fillna(df['amount'].median(), inplace=True)
df['category'].fillna(df['category'].mode()[0], inplace=True)
```
- 适用场景：数据近似对称分布（均值）或有偏分布（中位数）
- 风险：会缩小组内方差，低估不确定性

**策略3：填充——前向/后向**
```python
# 时间序列缺失值的前向填充
df['amount'].fillna(method='ffill', inplace=True)
```
- 适用场景：时间序列数据
- 风险：假设"最近的值最相关"，对突变不敏感

**策略4：保留缺失作为信号**
```python
# 创建缺失指示变量
df['amount_missing'] = df['amount'].isnull().astype(int)
```
- 适用场景：缺失本身可能有业务含义（如"未填写收入"可能暗示低收入人群）
- 优势：不丢失信息，让模型自己学习缺失的含义

**策略决策树**：
```
缺失率 < 1% → 删除行
缺失率 1%-10% → 填充（中位数/众数）或保留为信号
缺失率 10%-50% → 填充或保留为信号 + 缺失指示变量
缺失率 > 50% → 删除该列，或只保留缺失指示变量
```

### 2.2 异常值检测与处理

**为什么重要**：一个极端异常值可以毁掉你的均值、标准差、相关系数——所有基于这些统计量的结论都会出错。

**检测方法1：3-Sigma法则**
```python
mean = df['amount'].mean()
std = df['amount'].std()
outliers = df[abs(df['amount'] - mean) > 3 * std]
```
- 假设数据服从正态分布
- 问题：对偏态分布效果差（本身就是基于均值和标准差的）

**检测方法2：IQR(四分位距)法——推荐**
```python
Q1 = df['amount'].quantile(0.25)
Q3 = df['amount'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = df[(df['amount'] < lower) | (df['amount'] > upper)]
```
- 不依赖分布假设，更稳健
- 1.5是常用系数，增大更宽松，减小更严格

**检测方法3：业务规则**
```python
# 金额不能为负数，年龄不能超过120
invalid = df[(df['amount'] < 0) | (df['age'] > 120)]
```
- 最可靠的方法——你知道什么显然不可能

**处理异常值**：
- 显然错误（如负数金额）→ 删除或修正
- 真实但不典型（如VIP一次性大额采购）→ 保留但分段分析
- 不确定 → 标记后先保留，在分析时做"有异常值"和"无异常值"两个版本的对比

### 2.3 数据类型转换与格式统一

**为什么重要**：类型错误会导致静默失败——代码不报错但结果完全错误。

**常见转换**：
```python
# 字符串转日期
df['order_date'] = pd.to_datetime(df['order_date'])
# 字符串转数值
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
# 分类变量转category类型（节省内存）
df['category'] = df['category'].astype('category')
# 去除字符串字段的空白
df['city'] = df['city'].str.strip()
# 统一大小写
df['city'] = df['city'].str.lower()
```

### 2.4 描述性统计的正确解读

**为什么重要**：同样的均值，背后的分布可能完全不同——理解数据的"形状"比理解"平均值"更重要。

**核心统计量及其陷阱**：

| 统计量 | 计算公式 | 适用场景 | 陷阱 |
|--------|---------|---------|------|
| 均值(mean) | sum/n | 对称分布 | 受异常值影响极大 |
| 中位数(median) | 排序后中间值 | 偏态分布 | 对分布中间敏感，两端不敏感 |
| 标准差(std) | sqrt(Σ(x-μ)²/n) | 衡量离散程度 | 同样受异常值影响 |
| IQR | Q3-Q1 | 稳健的离散度 | 不反映两端的极端情况 |
| 偏度(skewness) | E[(X-μ)³]/σ³ | 判断分布对称性 | 正值意味着右偏（长尾在右） |
| 峰度(kurtosis) | E[(X-μ)⁴]/σ⁴ | 判断尾部厚度 | 高值意味着厚尾（更多极端值） |

**解读技巧**：
- 均值 >> 中位数 → 右偏分布（有少数大额订单拉高了均值）
- 均值 << 中位数 → 左偏分布（有少数极低值拉低了均值）
- 标准差 > 均值 → 数据波动非常大，均值本身参考价值有限

### 2.5 相关性分析的正确使用

**为什么重要**：相关性是很多分析方法的基础，但也是最容易被误读的统计概念之一。

**Pearson相关系数**：
```python
corr_matrix = df[['amount', 'order_count', 'days_active']].corr()
```
- 测量线性关系的强度
- 范围：[-1, 1]，0表示无线性关系
- 陷阱1：r=0不一定意味着两个变量无关——它们可能是非线性关系（如U型曲线）
- 陷阱2：r=0.8不等于"X导致Y"——可能是反向因果，也可能是第三个变量Z同时影响了X和Y
- 陷阱3：异常值可以凭空制造或摧毁一个相关性

**Spearman秩相关系数**：
```python
spearman_corr = df[['amount', 'order_count']].corr(method='spearman')
```
- 测量单调关系（不一定是线性）
- 对异常值更稳健
- 适合有序分类变量

### 2.6 从EDA到假设的转化

**为什么重要**：EDA的最终产出不是一堆图表，而是3-5个清晰可验证的假设。

**假设的标准格式**：
1. "X群体的Y指标显著高于/低于Z群体的Y指标"
2. "A因素和B因素之间存在正/负相关关系"
3. "在不同条件下，C指标的分布存在显著差异"

**从观察到假设的转化示例**：
- 观察到：箱线图显示paid渠道的用户客单价似乎更高
- 转化为假设："paid渠道的客单价中位数显著高于organic渠道"
- 验证方法：Mann-Whitney U检验（两组独立样本的非参数检验）

### 2.7 EDA可视化速查表

| 目标 | 图表类型 | Python代码 |
|------|---------|------------|
| 看一个数值变量的分布 | 直方图+密度曲线 | sns.histplot(data, kde=True) |
| 看一个分类变量的分布 | 柱状图(计数图) | sns.countplot(data, x='category') |
| 看数值vs数值的关系 | 散点图 | sns.scatterplot(data, x='a', y='b') |
| 看数值vs分类的对比 | 箱线图 | sns.boxplot(data, x='cat', y='val') |
| 看两个分类变量的交叉 | 热力图 | sns.heatmap(pd.crosstab(df['a'], df['b'])) |
| 看多个数值的相关性 | 相关矩阵热力图 | sns.heatmap(df.corr(), annot=True) |
| 看时间序列趋势 | 折线图 | df.groupby('date')['val'].sum().plot() |

### 2.8 数据清洗的自动化脚本模板

**为什么重要**：每次手动清洗数据是低效且不可重复的。建立自己的清洗模板，每次基于模板修改。

## 区块3：动手练习

### 步骤0：加载数据并做初始嗅探

In [ ]:
# 步骤0：加载数据并做完整的初始嗅探
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(42)

# ============================================================
# 生成带"脏数据"的数据集（故意加入缺失值、异常值、重复值）
# ============================================================
n = 10000
df_raw = pd.DataFrame({
    'order_id': range(1, n + 1),
    'user_id': np.random.choice(range(1, 1001), n),
    'order_date': pd.to_datetime(np.random.choice(
        pd.date_range('2026-01-01', '2026-06-30'), n)),
    'category': np.random.choice(['服装', '数码', '食品', '美妆', '家居', None], n,
                                 p=[0.29, 0.14, 0.24, 0.19, 0.09, 0.05]),
    'amount': np.round(np.random.lognormal(4, 0.7, n), 2),
    'channel': np.random.choice(['organic', 'paid', 'social', 'referral'], n),
})

# 故意制造一些问题
df_raw.loc[np.random.choice(n, 50), 'amount'] = np.nan  # 50个缺失值
df_raw.loc[np.random.choice(n, 10), 'amount'] = -999  # 10个异常值(负数)
df_raw.loc[np.random.choice(n, 3), 'amount'] = 999999  # 3个极端异常值
df_raw = pd.concat([df_raw, df_raw.sample(20)], ignore_index=True)  # 20条重复
df_raw.loc[np.random.choice(len(df_raw), 30), 'channel'] = 'Organic'  # 30处大小写不一致

print("=" * 60)
print("STEP 0: 初始数据嗅探报告")
print("=" * 60)
print(f"\n[基本信息]")
print(f"  行数: {len(df_raw):,}")
print(f"  列数: {len(df_raw.columns)}")
print(f"  列名: {list(df_raw.columns)}")
print(f"\n[数据类型]")
print(df_raw.dtypes)
print(f"\n[缺失值]")
missing_info = pd.DataFrame({
    '缺失数': df_raw.isnull().sum(),
    '缺失率': (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
})
print(missing_info[missing_info['缺失数'] > 0])
print(f"\n[重复行]")
print(f"  重复行数: {df_raw.duplicated().sum()}")
print(f"\n[基础统计量]")
print(df_raw.describe())

### 步骤1：数据清洗——逐一解决问题

In [ ]:
# 步骤1：系统化数据清洗
df = df_raw.copy()

# 1.1 去重
dup_before = len(df)
df = df.drop_duplicates()
print(f"[去重] {dup_before}行 → {len(df)}行 (删除{dup_before - len(df)}条重复)")

# 1.2 处理缺失值——category有5%缺失
cat_missing = df['category'].isnull().sum()
df['category'] = df['category'].fillna('未知')
print(f"[缺失值-category] {cat_missing}条缺失 → 填充为'未知'")

# 1.3 处理缺失值——amount有50个缺失
amt_missing = df['amount'].isnull().sum()
df['amount_missing_flag'] = df['amount'].isnull().astype(int)  # 保留缺失信号
df['amount'] = df['amount'].fillna(df['amount'].median())
print(f"[缺失值-amount] {amt_missing}条缺失 → 中位数填充 + 创建缺失标记列")

# 1.4 处理异常值——负数和极端值
negative_mask = df['amount'] < 0
print(f"[异常值-负数] {negative_mask.sum()}条 → 删除")
df = df[~negative_mask]

# IQR法检测极端异常值
Q1, Q3 = df['amount'].quantile(0.25), df['amount'].quantile(0.75)
IQR = Q3 - Q1
upper = Q3 + 3 * IQR  # 使用3倍IQR，只标记最极端的
extreme = df['amount'] > upper
print(f"[异常值-极端高值] {extreme.sum()}条(IQR法, >Q3+3*IQR) → 标记但保留(可能是VIP大额订单)")
df['is_extreme_amount'] = extreme.astype(int)

# 1.5 统一格式——channel大小写
df['channel'] = df['channel'].str.lower().str.strip()
print(f"[格式统一] channel列: 统一为小写+去除空格")

# 检查channel的唯一值
print(f"  channel唯一值: {df['channel'].unique()}")

# 1.6 数据类型最终确认
df['category'] = df['category'].astype('category')
df['channel'] = df['channel'].astype('category')
print(f"[类型优化] category和channel转换为category类型")
print(f"  内存减少: {(df_raw.memory_usage(deep=True).sum() - df.memory_usage(deep=True).sum()) / 1024:.1f} KB")

print(f"\n清洗后数据集: {len(df)}行 × {len(df.columns)}列")
print(f"最终缺失值: {df.isnull().sum().sum()}")

### 步骤2：单变量EDA——逐个理解每个字段

In [ ]:
# 步骤2：单变量EDA——理解每个"列"
print("=" * 50)
print("单变量EDA")
print("=" * 50)

# 2.1 数值变量：amount
print(f"\n[金额分布]")
amount_stats = df['amount'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99])
print(amount_stats.round(2))
print(f"  偏度: {df['amount'].skew():.2f} (正值=右偏/长尾向右)")
print(f"  峰度: {df['amount'].kurtosis():.2f} (正值=厚尾)")

# 2.2 分类变量：category
print(f"\n[品类分布]")
cat_dist = df['category'].value_counts()
for cat, cnt in cat_dist.items():
    print(f"  {cat}: {cnt} ({cnt/len(df)*100:.1f}%)")

# 2.3 分类变量：channel
print(f"\n[渠道分布]")
ch_dist = df['channel'].value_counts()
for ch, cnt in ch_dist.items():
    print(f"  {ch}: {cnt} ({cnt/len(df)*100:.1f}%)")

# 2.4 日期变量
print(f"\n[日期范围]")
print(f"  最早: {df['order_date'].min().date()}")
print(f"  最晚: {df['order_date'].max().date()}")
print(f"  跨度: {(df['order_date'].max() - df['order_date'].min()).days} 天")

# 月度订单分布
monthly_orders = df.groupby(df['order_date'].dt.to_period('M')).size()
print(f"\n[月度订单分布]")
for month, cnt in monthly_orders.items():
    print(f"  {month}: {cnt}单")

### 步骤3：多变量EDA——发现关系与模式

In [ ]:
# 步骤3：双变量与多变量EDA——发现关系
print("=" * 50)
print("双变量与多变量EDA")
print("=" * 50)

# 3.1 各品类金额对比
print(f"\n[品类 × 金额]")
cat_amount = df.groupby('category', observed=True).agg(
    订单数=('amount', 'count'),
    平均金额=('amount', 'mean'),
    中位数金额=('amount', 'median'),
    总金额=('amount', 'sum')
).round(2)
print(cat_amount)

# 3.2 各渠道金额对比
print(f"\n[渠道 × 金额]")
ch_amount = df.groupby('channel', observed=True).agg(
    订单数=('amount', 'count'),
    平均金额=('amount', 'mean'),
    中位数金额=('amount', 'median'),
    总金额=('amount', 'sum')
).round(2)
print(ch_amount)

# 3.3 品类×渠道交叉
print(f"\n[品类 × 渠道 交叉分布]")
cross_tab = pd.crosstab(df['category'], df['channel'], normalize='columns') * 100
print(cross_tab.round(1))

# 3.4 金额与月份的关系
df['month'] = df['order_date'].dt.month
print(f"\n[月份 × 平均金额]")
monthly_avg = df.groupby('month')['amount'].agg(['mean', 'median', 'count']).round(2)
print(monthly_avg)

# 3.5 判断是否需要统计检验
from scipy import stats
print(f"\n[统计检验——渠道间金额差异是否显著？]")
organic_amt = df[df['channel'] == 'organic']['amount']
paid_amt = df[df['channel'] == 'paid']['amount']
stat, p_value = stats.mannwhitneyu(organic_amt, paid_amt, alternative='two-sided')
print(f"  organic vs paid — Mann-Whitney U检验:")
print(f"  统计量: {stat:.0f}")
print(f"  p值: {p_value:.4f}")
print(f"  结论: {'显著差异' if p_value < 0.05 else '无显著差异'} (α=0.05)")

### 步骤4：挑战练习——从EDA到假设

In [ ]:
# 步骤4：挑战练习——基于EDA提出3个可验证的假设
print("=" * 50)
print("挑战练习：从EDA到假设")
print("=" * 50)

print("""
基于以上EDA，我提出以下3个假设，将在后续分析中验证：

假设1: 品类差异
  观察: 数码品类的平均金额显著高于其他品类
  假设: 数码品类的平均客单价 > 整体平均客单价的1.5倍
  验证方法: 单样本t检验或bootstrap置信区间
  业务含义: 如果成立→数码用户是"高客单价但低频次"群体，
            运营策略应侧重"推动复购"而非"提升客单价"

假设2: 渠道质量
  观察: organic渠道的订单数最多，但人均消费可能较低
  假设: paid渠道的客单价中位数显著高于organic渠道
  验证方法: Mann-Whitney U检验（已在上面完成）
  业务含义: 如果成立→paid渠道虽然成本高，但用户质量高，
            ROI可能仍然是正的（要结合获客成本计算）

假设3: 时间趋势
  观察: 月度订单数有波动
  假设: 存在显著的季节性模式（某几个月订单量系统性更高）
  验证方法: 时间序列分解或ANOVA检验各月均值是否相等
  业务含义: 如果成立→旺季提前备货/加推广，淡季做促销/清库存
""")

print("\n提示: 这些假设将在W11D4-5的深度分析中逐一验证。")
print("EDA的价值不在于'看到数据'，而在于'知道该往哪个方向深挖'。")

## 区块4：业务场景应用

### 场景1：发现"异常值"原来是金矿

**问题**：做EDA时发现订单金额top 1%的用户贡献了25%的GMV，boxplot上他们是"异常值"。要不要删除？

**分析**：
- 这些"异常值"不是数据错误——他们是VIP大客户
- 删除他们会严重低估平台的GMV
- 应该把他们单独拿出来分析：这些人有什么共同特征？怎么让更多人变成VIP？

**决策**：
- 保留所有数据
- 在做整体分析时，如果均值被极端值拉高，同时报告均值和中位数
- 单独做一个"VIP用户画像"的侧栏分析

**教训**："异常"不等于"错误"——有时异常本身就是最重要的发现。

### 场景2：缺失值"说"出了用户的秘密

**问题**：用户表中"收入"字段有30%缺失。直接删除这些用户会损失大量样本，填充均值又可能引入偏差。

**分析**：
- 先做一个"收入缺失 vs 收入未缺失"的对比
- 发现：收入缺失的用户，平均消费金额显著低于收入填写的用户
- 推测：不填收入的用户更可能是低收入群体（隐私顾虑）

**决策**：
- 保留缺失值，并创建一个"收入是否缺失"的二元特征
- 这个特征本身就有预测力——"收入缺失"可能意味着"低消费意愿"

**教训**：缺失值不一定是"坏数据"——它可能是用户行为的一个信号。

### 场景3：数据清洗和业务理解的关系

**问题**：发现某个品类在6月的销量暴跌80%。是数据问题还是业务问题？

**排查步骤**：
1. 检查数据口径是否变了（如品类重新分类导致部分商品归到其他品类）
2. 检查是否有系统故障导致数据丢失
3. 如果以上都不是——和业务方确认，6月是否发生了什么事情（如竞品大促、供应链断货）

**关键认知**：数据清洗不只是"技术活"——你需要业务知识来判断"这个数据对不对"。最好的数据分析师不是最会写代码的，而是最懂业务的。

## 区块5：分级检验清单

### 初级检验（必须能独立完成的）
- [ ] 能用df.info()、df.describe()、df.isnull().sum()完成基本数据嗅探
- [ ] 能用drop_duplicates()去重、fillna()填充缺失值
- [ ] 能用IQR法或3-sigma法检测异常值
- [ ] 能画出至少3种EDA图表（直方图、箱线图、柱状图）

### 中级检验（能解释给同事听的原理）
- [ ] 能解释缺失值的4种处理策略及其决策框架
- [ ] 能解释IQR法为什么比3-sigma法更稳健
- [ ] 能区分"数据错误型异常值"和"真实但极端型异常值"，并给出不同的处理策略
- [ ] 能从EDA中发现至少3个值得深挖的模式或关系

### 高级检验（能举一反三的迁移能力）
- [ ] 能设计完整的数据质量检查方案（覆盖6个质量维度）
- [ ] 能将EDA发现转化为可验证的统计假设
- [ ] 能用统计检验（t检验/Mann-Whitney/卡方检验）验证EDA中的观察是否显著
- [ ] 能判断缺失值的产生机制（MCAR/MAR/MNAR）并选择对应的处理策略

**答案提示**：
- MCAR(完全随机缺失): 缺失与任何变量无关 → 可直接删除
- MAR(随机缺失): 缺失与其他观测变量有关 → 需要纳入模型
- MNAR(非随机缺失): 缺失与缺失值本身有关 → 最复杂，需专门建模

## 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 前置依赖** | W11D1数据获取 | 今天直接处理昨天生成/获取的数据 |
| **← 前置依赖** | W3D6数据嗅探 | 更系统化的数据质量检查流程 |
| **→ 后续用途** | W11D3特征工程 | 清洗后的数据才能做特征工程 |
| **→ 后续用途** | W11D4-5深度分析 | 错误的数据=错误的分析结论 |
| **业务连接** | 所有数据分析工作 | EDA是每天开始分析前的第一步 |

### 横向连接：与统计方法的关联

**与假设检验的连接**：
- EDA提出假设 → 统计检验验证假设
- 注意：不要对同一个数据先做EDA"找模式"然后又做假设检验"验证"——这会导致p-hacking

**与特征工程的连接**：
- EDA中发现的强预测变量 → 重点做特征工程
- EDA中发现的非线性关系 → 创建交互项或多项式特征

**与机器学习的连接**：
- 缺失值处理策略影响模型性能
- 异常值处理影响模型训练稳定性
- EDA帮助选择合适的数据变换（log变换、标准化等）

## 区块7：当日小结

> **永远先做EDA再做分析。数据质量有六个维度：完整性、唯一性、一致性、准确性、有效性、时效性。缺失值处理不能一刀切——要根据缺失率和缺失机制选择策略。异常值不一定是错误——有时异常本身就是最大的发现。EDA的产出不是一堆图表，而是3-5个可验证的假设。花2小时把数据看透，能帮你省掉后面20小时的"方向错误"。**

**三条金句带回家**：
1. EDA的三层次：单变量（看分布）→ 双变量（看关系）→ 多变量（看交互），每一层都在回答更深入的问题
2. 缺失值策略决策树：<1%删除、1-10%填充、10-50%填充+信号、>50%删除列——但永远先问"为什么缺失"
3. "异常值"不等于"错误值"——VIP大客户的订单在boxplot上是异常值，在业务上是金矿

### 课后拓展

1. **练习**：用IQR法标记异常值后，对比"含异常值"和"去除异常值"两种情况下均值的变化，体会异常值对统计量的影响

2. **思考**：如果你的数据有50%的缺失值，删除会损失一半样本，填充会引入巨大偏差——你会怎么办？

3. **预习**：明天要做的特征工程将基于今天的发现——每个EDA中发现的关系，都可以转化为一个特征

4. **面试准备**：准备一个"你在数据清洗中遇到过最棘手的问题"的回答。好的答案是：具体问题 + 你的分析过程 + 解决方案 + 业务影响。